In [1]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder
import joblib

import rsutils.modify_bands

In [2]:
TRAINING_DATA_FOLDERPATH = '/Users/nikhilsrajan/NASA-Harvest/project/fetch_satdata/data/at_eurocrops/training_data'
MODEL_FILEPATH = '/Users/nikhilsrajan/NASA-Harvest/project/fetch_satdata/data/at_eurocrops/rf_model.joblib'

In [3]:
training_data = np.load(os.path.join(TRAINING_DATA_FOLDERPATH, 'data.npy'))
ids = np.load(os.path.join(TRAINING_DATA_FOLDERPATH, 'ids.npy'))
labels = np.load(os.path.join(TRAINING_DATA_FOLDERPATH, 'labels.npy'))
training_metadata = np.load(os.path.join(TRAINING_DATA_FOLDERPATH, 'metadata.pickle.npy'), allow_pickle=True)[()]

In [4]:
def median_per_id(ids, data, labels):
    _, n_ts, n_bands = data.shape
    
    unique_ids, inverse = np.unique(ids, return_inverse=True)
    n_ids = len(unique_ids)

    data_median = np.zeros(shape=(n_ids, n_ts, n_bands), dtype=data.dtype)
    labels_median = np.zeros(n_ids, dtype=labels.dtype)
    
    for i in range(n_ids):
        data_median[i] = np.nanmedian(data[inverse == i], axis=0)
        labels_median[i] = labels[inverse == i][0]

    ids_median = unique_ids

    return ids_median, data_median, labels_median    

In [ ]:
sentinel_2_l2a_bands = ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B11', 'B12']

bands = training_metadata.get('bands')
band_indices = {band: index for index, band in enumerate(bands)}

modified_training_data, modified_band_indices = \
rsutils.modify_bands.modify_bands(
    bands = np.expand_dims(training_data, axis=(2, 3)).astype(float),
    band_indices = band_indices,
    sequence = [
        (rsutils.modify_bands.mask_invalid_and_interpolate, {}),
        (rsutils.modify_bands.compute_bands, dict(bands_to_compute = ['NDVI', 'NDRE', 'GCVI', 'SAVI'])),
        (rsutils.modify_bands.remove_bands, dict(bands_to_remove = sentinel_2_l2a_bands))
        # (rsutils.modify_bands.scale_bands, dict(bands_to_scale = sentinel_2_l2a_bands, std = 10000))
    ]
)

modified_training_data = np.squeeze(modified_training_data)

In [ ]:
le = LabelEncoder()

clf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

# ---- reshape ----
n_ids, n_timestamps, n_bands = modified_training_data.shape
X = modified_training_data.reshape(n_ids, n_timestamps * n_bands)
y = labels
y_encoded = le.fit_transform(y)

# ---- train/test split ----
X_train, X_test, y_train_encoded, y_test_encoded = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# ---- model ----

clf.fit(X_train, y_train_encoded)

# ---- evaluation ----
y_pred_encoded = clf.predict(X_test)

y_pred = le.inverse_transform(y_pred_encoded)
y_test = le.inverse_transform(y_test_encoded)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
joblib.dump((clf, le), MODEL_FILEPATH)